## NCAA Seed Prediction: Ensemble (v2) — Accuracy priority

This notebook is tuned for **maximum accuracy** (no speed shortcuts):
1. **Feature engineering**: WL parsing, season encoding, NET-derived and binned features, quadrant ratios, conference mean seed.
2. **Optuna-tuned GBMs**: LightGBM, XGBoost, CatBoost with **200 trials** each and wide search space; **season-based CV** (GroupKFold).
3. **AutoGluon**: 10 min, best_quality preset.
4. **Stacking**: Ridge meta-learner on out-of-fold predictions for optimal blend; then 15% AutoGluon if used. Fallback: inverse-CV-RMSE weighted average.
5. **Output**: Rank-by-season (1-68 no duplicates per season), optional official-seeds overlay; saves `../Output/submission_ensemble.csv`. **Config:** `USE_RANK_BY_SEASON`, `USE_OFFICIAL_SEEDS`.

In [ ]:
# Install required packages (run once; safe to re-run).
import subprocess
import sys
def _pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + list(args))
_pip("lightgbm", "xgboost", "catboost", "optuna", "scikit-learn")
try:
    _pip("autogluon.tabular")
except Exception:
    print("autogluon.tabular install failed; set USE_AUTOGLUON=False to skip.")
print("Core packages installed.")

In [ ]:
# Colab: mount Drive and set DATA_DIR to the folder with the 3 CSVs.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = "/content/drive/MyDrive/Kaggle NCAA competition Deadline 15th March/final-four-analytics-challenge-26"
except Exception:
    DATA_DIR = "../final-four-analytics-challenge-26"

import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
import lightgbm as lgb
import xgboost as xgb
try:
    import catboost as cb
except ImportError:
    cb = None

RANDOM_STATE = 42
N_FOLDS = 5
OPTUNA_TRIALS = 200  # Accuracy priority: more trials for better hyperparameters
USE_AUTOGLUON = True
AUTOGLUON_TIME_LIMIT = 600  # 10 min for best_quality
AUTOGLUON_PRESET = "best_quality"
USE_SEASON_CV = True
USE_CATBOOST = True
USE_STACKING = True  # Ridge meta-learner on OOF predictions for best blend
USE_RANK_BY_SEASON = True  # Assign seeds 1-68 per season (no duplicates); helps evaluation
USE_OFFICIAL_SEEDS = False  # Overlay known seeds from official_seeds.py if allowed by rules

In [ ]:
def _path(name_2_0, name_default):
    for base in [DATA_DIR, os.path.join(DATA_DIR, "Data")]:
        p2 = os.path.join(base, name_2_0)
        p1 = os.path.join(base, name_default)
        if os.path.exists(p2):
            return p2
        if os.path.exists(p1):
            return p1
    return os.path.join(DATA_DIR, name_2_0)

train = pd.read_csv(_path("NCAA_Seed_Training_Set2.0.csv", "NCAA_Seed_Training_Set.csv"))
test = pd.read_csv(_path("NCAA_Seed_Test_Set2.0.csv", "NCAA_Seed_Test_Set.csv"))
sub = pd.read_csv(_path("submission_template2.0.csv", "submission_template.csv"))
print("Train:", train.shape, "Test:", test.shape)

In [ ]:
MONTH_TO_NUM = {"Jan": 1, "Feb": 2, "Mar": 3, "Apr": 4, "May": 5, "Jun": 6,
                "Jul": 7, "Aug": 8, "Sep": 9, "Oct": 10, "Nov": 11, "Dec": 12}

def parse_wl(val):
    if pd.isna(val) or val == "" or str(val).strip() == "0-0":
        return np.nan, np.nan, np.nan
    s = str(val).strip()
    parts = s.split("-")
    if len(parts) != 2:
        return np.nan, np.nan, np.nan
    def to_num(x):
        x = x.strip()
        if x in MONTH_TO_NUM:
            return MONTH_TO_NUM[x]
        try:
            return int(x)
        except ValueError:
            return np.nan
    w, l = to_num(parts[0]), to_num(parts[1])
    if np.isnan(w) or np.isnan(l):
        return np.nan, np.nan, np.nan
    total = w + l
    pct = w / total if total > 0 else np.nan
    return w, l, pct

def add_wl_features(df, col):
    if col not in df.columns:
        return df
    unpacked = list(zip(*[parse_wl(x) for x in df[col]]))
    if not unpacked:
        return df
    df = df.copy()
    df[f"{col}_w"], df[f"{col}_l"], df[f"{col}_pct"] = unpacked[0], unpacked[1], unpacked[2]
    return df

wl_cols = ["WL", "Conf.Record", "Non-ConferenceRecord", "RoadWL", "Quadrant1", "Quadrant2", "Quadrant3", "Quadrant4"]
for col in wl_cols:
    train = add_wl_features(train, col)
    test = add_wl_features(test, col)

In [ ]:
# Season as numeric (e.g. 2020-21 -> 2021)
def season_to_year(s):
    if pd.isna(s):
        return np.nan
    s = str(s).strip()
    if "-" in s:
        return int(s.split("-")[1])
    return np.nan

train["SeasonYear"] = train["Season"].map(season_to_year)
test["SeasonYear"] = test["Season"].map(season_to_year)

# NET-derived: inverse rank (higher rank = worse, so 1/rank or log; also rank/364 for scale)
for df in [train, test]:
    df["NET_inv"] = 1.0 / (df["NET Rank"].clip(lower=1))
    df["NET_log"] = np.log1p(df["NET Rank"].fillna(400))
    df["NETSOS_inv"] = 1.0 / (df["NETSOS"].clip(lower=1))
    if "NETNonConfSOS" in df.columns:
        df["NETNonConfSOS_inv"] = 1.0 / (df["NETNonConfSOS"].clip(lower=1))

# Quadrant totals and Q1 share (strength-of-schedule signal)
quad_w = [c for c in train.columns if c.endswith("_w") and "Quadrant" in c]
if quad_w:
    train["QuadW_total"] = train[quad_w].sum(axis=1)
    test["QuadW_total"] = test[quad_w].sum(axis=1)
    if "Quadrant1_w" in train.columns:
        train["Q1_share"] = train["Quadrant1_w"] / (train["QuadW_total"].replace(0, np.nan))
        test["Q1_share"] = test["Quadrant1_w"] / (test["QuadW_total"].replace(0, np.nan))

# Conference strength: mean seed by conference (from training teams with seeds)
train_with_seed = train.dropna(subset=["Overall Seed"])
conf_mean_seed = train_with_seed.groupby("Conference")["Overall Seed"].mean().astype(float).to_dict()
train["Conf_mean_seed"] = train["Conference"].map(conf_mean_seed)
test["Conf_mean_seed"] = test["Conference"].map(conf_mean_seed)

# NET rank bins (top 16, 17-32, 33-68, 69-200, 200+)
for df in [train, test]:
    r = df["NET Rank"].fillna(400)
    df["NET_bin"] = np.where(r <= 16, 1, np.where(r <= 32, 2, np.where(r <= 68, 3, np.where(r <= 200, 4, 5))))

# Interaction: NET * win pct (strong teams: low NET, high WL_pct)
for df in [train, test]:
    df["NET_x_WLpct"] = (df["NET Rank"].fillna(300)) * (df["WL_pct"].fillna(0.5))

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer

num_cols = [
    "NET Rank", "PrevNET", "AvgOppNETRank", "AvgOppNET",
    "NETSOS", "NETNonConfSOS",
    "NET_inv", "NET_log", "NETSOS_inv", "NETNonConfSOS_inv",
    "SeasonYear",
    "WL_w", "WL_l", "WL_pct",
    "Conf.Record_w", "Conf.Record_l", "Conf.Record_pct",
    "Non-ConferenceRecord_w", "Non-ConferenceRecord_l", "Non-ConferenceRecord_pct",
    "RoadWL_w", "RoadWL_l", "RoadWL_pct",
    "Quadrant1_w", "Quadrant1_l", "Quadrant1_pct",
    "Quadrant2_w", "Quadrant2_l", "Quadrant2_pct",
    "Quadrant3_w", "Quadrant3_l", "Quadrant3_pct",
    "Quadrant4_w", "Quadrant4_l", "Quadrant4_pct",
    "QuadW_total", "Q1_share",
    "Conf_mean_seed", "NET_bin", "NET_x_WLpct",
]
num_cols = [c for c in num_cols if c in train.columns and c in test.columns]

for cat in ["Conference", "Bid Type"]:
    if cat not in train.columns:
        continue
    all_vals = pd.concat([train[cat], test[cat]], ignore_index=True).astype(str).fillna("__NA__")
    le = LabelEncoder()
    le.fit(all_vals.unique())
    train[f"{cat}_enc"] = le.transform(train[cat].astype(str).fillna("__NA__"))
    test[f"{cat}_enc"] = test[cat].astype(str).fillna("__NA__").map(
        lambda x: le.transform([x])[0] if x in le.classes_ else -1
    )
    num_cols.append(f"{cat}_enc")

feature_cols = [c for c in num_cols if c in train.columns and c in test.columns]
print("N features:", len(feature_cols))

In [ ]:
train_seed = train.dropna(subset=["Overall Seed"]).copy()
train_seed["Overall Seed"] = train_seed["Overall Seed"].astype(int)
X = train_seed[feature_cols]
y = train_seed["Overall Seed"]
X_test = test[feature_cols]
groups = train_seed["Season"].values  # For season-based CV

imp = SimpleImputer(strategy="median")
X_imp = imp.fit_transform(X)
X_test_imp = imp.transform(X_test)
print("Train samples:", len(y), "Target range:", int(y.min()), "-", int(y.max()))
print("Seasons in train:", pd.Series(groups).nunique())

In [ ]:
import optuna
from sklearn.model_selection import KFold, GroupKFold, cross_val_predict
from sklearn.metrics import mean_squared_error

optuna.logging.set_verbosity(optuna.logging.WARNING)

def lgb_objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 400, 1500),
        "max_depth": trial.suggest_int("max_depth", 5, 18),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 31, 200),
        "min_child_samples": trial.suggest_int("min_child_samples", 2, 50),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
        "random_state": RANDOM_STATE,
        "verbosity": -1,
        "n_jobs": -1,
    }
    cv = GroupKFold(n_splits=N_FOLDS) if USE_SEASON_CV else KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    preds = cross_val_predict(
        lgb.LGBMRegressor(**params),
        X_imp, y, cv=cv, groups=groups if USE_SEASON_CV else None, n_jobs=1
    )
    return np.sqrt(mean_squared_error(y, preds))

study_lgb = optuna.create_study(direction="minimize")
study_lgb.optimize(lgb_objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)
print("LightGBM best CV RMSE:", round(study_lgb.best_value, 4))
print("Best params:", study_lgb.best_params)

In [ ]:
def xgb_objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 400, 1500),
        "max_depth": trial.suggest_int("max_depth", 5, 16),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 30),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
    }
    cv = GroupKFold(n_splits=N_FOLDS) if USE_SEASON_CV else KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    preds = cross_val_predict(
        xgb.XGBRegressor(**params),
        X_imp, y, cv=cv, groups=groups if USE_SEASON_CV else None, n_jobs=1
    )
    return np.sqrt(mean_squared_error(y, preds))

study_xgb = optuna.create_study(direction="minimize")
study_xgb.optimize(xgb_objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)
print("XGBoost best CV RMSE:", round(study_xgb.best_value, 4))
print("Best params:", study_xgb.best_params)

In [ ]:
# Optional: CatBoost with Optuna (pip install catboost)
study_cat = None
if USE_CATBOOST and cb is not None:
    def cat_objective(trial):
        params = {
            "iterations": trial.suggest_int("iterations", 400, 1500),
            "depth": trial.suggest_int("depth", 5, 14),
            "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-4, 10.0, log=True),
            "random_seed": RANDOM_STATE,
            "verbose": 0,
        }
        cv = GroupKFold(n_splits=N_FOLDS) if USE_SEASON_CV else KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
        preds = cross_val_predict(
            cb.CatBoostRegressor(**params),
            X_imp, y, cv=cv, groups=groups if USE_SEASON_CV else None, n_jobs=1
        )
        return np.sqrt(mean_squared_error(y, preds))
    study_cat = optuna.create_study(direction="minimize")
    study_cat.optimize(cat_objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)
    print("CatBoost best CV RMSE:", round(study_cat.best_value, 4))
else:
    print("CatBoost skipped (USE_CATBOOST=False or catboost not installed).")

In [ ]:
# Train final models with best hyperparameters
model_lgb = lgb.LGBMRegressor(**study_lgb.best_params, random_state=RANDOM_STATE, verbosity=-1, n_jobs=-1)
model_xgb = xgb.XGBRegressor(**study_xgb.best_params, random_state=RANDOM_STATE, n_jobs=-1)

model_lgb.fit(X_imp, y)
model_xgb.fit(X_imp, y)

pred_lgb = model_lgb.predict(X_test_imp)
pred_xgb = model_xgb.predict(X_test_imp)
pred_cat = None
if study_cat is not None and cb is not None:
    model_cat = cb.CatBoostRegressor(**study_cat.best_params, random_seed=RANDOM_STATE, verbose=0)
    model_cat.fit(X_imp, y)
    pred_cat = model_cat.predict(X_test_imp)
    print("CatBoost test pred range:", pred_cat.min().round(2), "-", pred_cat.max().round(2))
print("LightGBM test pred range:", pred_lgb.min().round(2), "-", pred_lgb.max().round(2))
print("XGBoost test pred range:", pred_xgb.min().round(2), "-", pred_xgb.max().round(2))

In [ ]:
# Stacking: OOF predictions + Ridge meta-learner for optimal blend (accuracy priority)
from sklearn.linear_model import Ridge
meta_ridge = None
if USE_STACKING:
    cv = GroupKFold(n_splits=N_FOLDS) if USE_SEASON_CV else KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    oof_lgb = np.zeros(len(y))
    oof_xgb = np.zeros(len(y))
    oof_cat = np.zeros(len(y)) if pred_cat is not None else None
    for tr_idx, val_idx in cv.split(X_imp, y, groups=groups if USE_SEASON_CV else None):
        m_lgb = lgb.LGBMRegressor(**study_lgb.best_params, random_state=RANDOM_STATE, verbosity=-1, n_jobs=-1)
        m_xgb = xgb.XGBRegressor(**study_xgb.best_params, random_state=RANDOM_STATE, n_jobs=-1)
        m_lgb.fit(X_imp[tr_idx], y.iloc[tr_idx])
        m_xgb.fit(X_imp[tr_idx], y.iloc[tr_idx])
        oof_lgb[val_idx] = m_lgb.predict(X_imp[val_idx])
        oof_xgb[val_idx] = m_xgb.predict(X_imp[val_idx])
        if pred_cat is not None and cb is not None:
            m_cat = cb.CatBoostRegressor(**study_cat.best_params, random_seed=RANDOM_STATE, verbose=0)
            m_cat.fit(X_imp[tr_idx], y.iloc[tr_idx])
            oof_cat[val_idx] = m_cat.predict(X_imp[val_idx])
    X_meta = np.column_stack([oof_lgb, oof_xgb] + ([oof_cat] if oof_cat is not None else []))
    meta_ridge = Ridge(alpha=1.0, random_state=RANDOM_STATE).fit(X_meta, y)
    stack_rmse = np.sqrt(mean_squared_error(y, meta_ridge.predict(X_meta)))
    print("Stacking Ridge fit. OOF RMSE:", round(stack_rmse, 4))
else:
    print("Stacking disabled (USE_STACKING=False).")

In [ ]:
# Optional: AutoGluon (install with: pip install autogluon.tabular)
pred_ag = None
if USE_AUTOGLUON:
    try:
        from autogluon.tabular import TabularPredictor
        TARGET = "Overall Seed"
        train_ag = pd.DataFrame(X_imp, columns=feature_cols).copy()
        train_ag[TARGET] = y.values
        test_ag = pd.DataFrame(X_test_imp, columns=feature_cols)
        predictor = TabularPredictor(label=TARGET, problem_type="regression").fit(
            train_ag, time_limit=AUTOGLUON_TIME_LIMIT, presets=AUTOGLUON_PRESET
        )
        pred_ag = predictor.predict(test_ag).values
        print("AutoGluon fitted. Test pred range:", pred_ag.min().round(2), "-", pred_ag.max().round(2))
    except Exception as e:
        print("AutoGluon skipped:", e)
        pred_ag = None
else:
    print("AutoGluon disabled (USE_AUTOGLUON=False).")

In [ ]:
# Final blend: stacking (accuracy priority) or weighted average
if meta_ridge is not None:
    test_meta = np.column_stack([pred_lgb, pred_xgb] + ([pred_cat] if pred_cat is not None else []))
    pred_ensemble = meta_ridge.predict(test_meta)
    if pred_ag is not None:
        pred_ensemble = 0.85 * pred_ensemble + 0.15 * pred_ag
    print("Blend: Ridge stack" + (" + AutoGluon (15%)" if pred_ag is not None else ""))
else:
    inv_rmse_lgb = 1.0 / study_lgb.best_value
    inv_rmse_xgb = 1.0 / study_xgb.best_value
    weights = [inv_rmse_lgb, inv_rmse_xgb]
    preds_list = [pred_lgb, pred_xgb]
    if pred_cat is not None:
        weights.append(1.0 / study_cat.best_value)
        preds_list.append(pred_cat)
    if pred_ag is not None:
        weights.append(np.mean(weights))
        preds_list.append(pred_ag)
    weights = np.array(weights) / np.sum(weights)
    pred_ensemble = sum(w * p for w, p in zip(weights, preds_list))
    names = ["LGB", "XGB"] + (["Cat"] if pred_cat is not None else []) + (["AG"] if pred_ag is not None else [])
    print("Weights (" + ", ".join(names) + "):", [round(w, 3) for w in weights])
pred_ensemble = np.clip(np.round(pred_ensemble), 1, 68).astype(int)
print("Ensemble pred range:", pred_ensemble.min(), "-", pred_ensemble.max())

In [ ]:
# Build final seeds: optional rank-by-season (1-68 per season, no duplicates), then optional official overlay
if USE_RANK_BY_SEASON:
    df_seed = test[["RecordID", "Season"]].copy()
    df_seed["pred"] = pred_ensemble
    df_seed["seed"] = np.nan
    for season in df_seed["Season"].unique():
        idx = df_seed["Season"] == season
        r = df_seed.loc[idx, "pred"].rank(method="first", ascending=True).astype(int)
        df_seed.loc[idx, "seed"] = np.clip(r, 1, 68)
    record_to_pred = dict(zip(df_seed["RecordID"], df_seed["seed"].astype(int)))
else:
    record_to_pred = dict(zip(test["RecordID"], pred_ensemble))

if USE_OFFICIAL_SEEDS:
    try:
        import sys
        for d in [DATA_DIR, os.path.join(DATA_DIR, "Data")]:
            if os.path.exists(os.path.join(d, "official_seeds.py")):
                sys.path.insert(0, d)
                break
        from official_seeds import get_seed_lookup
        official = get_seed_lookup()
        def _norm(s): return " ".join(str(s).strip().split())
        n_overlay = 0
        for rid in sub["RecordID"]:
            row = test[test["RecordID"] == rid].iloc[0]
            k = (row["Season"], _norm(row["Team"]))
            if k in official:
                record_to_pred[rid] = official[k]
                n_overlay += 1
        print("Official seeds overlayed for", n_overlay, "rows.")
    except Exception as e:
        print("Official seeds skipped:", e)

sub_out = sub[["RecordID"]].copy()
sub_out["Overall Seed"] = sub_out["RecordID"].map(record_to_pred)
out_dir = os.path.join(os.path.dirname(DATA_DIR), "Output")
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, "submission_ensemble.csv")
sub_out.to_csv(out_path, index=False)
print("Saved:", out_path)
print(sub_out.head(10))

try:
    from google.colab import files
    files.download(out_path)
    print("Download started.")
except Exception:
    pass